In [54]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [55]:
df =pd.read_csv("../Data/Breast_Cancer.csv")

In [56]:
df.head()

,Age,Race,Marital Status,T Stage,N Stage,6th Stage,differentiate,Grade,A Stage,Tumor Size,Estrogen Status,Progesterone Status,Regional Node Examined,Reginol Node Positive,Survival Months,Status
0,68,White,Married,T1,N1,IIA,Poorly differentiated,3,Regional,4,Positive,Positive,24,1,60,Alive
1,50,White,Married,T2,N2,IIIA,Moderately differentiated,2,Regional,35,Positive,Positive,14,5,62,Alive
2,58,White,Divorced,T3,N3,IIIC,Moderately differentiated,2,Regional,63,Positive,Positive,14,7,75,Alive
3,58,White,Married,T1,N1,IIA,Poorly differentiated,3,Regional,18,Positive,Positive,2,1,84,Alive
4,47,White,Married,T2,N1,IIB,Poorly differentiated,3,Regional,41,Positive,Positive,3,1,50,Alive


In [57]:
df.columns

Index(['Age', 'Race', 'Marital Status', 'T Stage ', 'N Stage', '6th Stage',
       'differentiate', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status',
       'Progesterone Status', 'Regional Node Examined',
       'Reginol Node Positive', 'Survival Months', 'Status'],
      dtype='object')

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4024 entries, 0 to 4023
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Age                     4024 non-null   int64 
 1   Race                    4024 non-null   object
 2   Marital Status          4024 non-null   object
 3   T Stage                 4024 non-null   object
 4   N Stage                 4024 non-null   object
 5   6th Stage               4024 non-null   object
 6   differentiate           4024 non-null   object
 7   Grade                   4024 non-null   object
 8   A Stage                 4024 non-null   object
 9   Tumor Size              4024 non-null   int64 
 10  Estrogen Status         4024 non-null   object
 11  Progesterone Status     4024 non-null   object
 12  Regional Node Examined  4024 non-null   int64 
 13  Reginol Node Positive   4024 non-null   int64 
 14  Survival Months         4024 non-null   int64 
 15  Stat

In [59]:
print(df.groupby("Marital Status")["Status"].apply(lambda s: (s == "Dead").mean()))
print(df["Marital Status"].value_counts())
print(df.groupby("Marital Status")["Age"].mean())

Marital Status
Divorced     0.185185
Married      0.135452
Separated    0.333333
Single       0.169106
Widowed      0.208511
Name: Status, dtype: float64
Marital Status
Married      2643
Single        615
Divorced      486
Widowed       235
Separated      45
Name: count, dtype: int64
Marital Status
Divorced     55.131687
Married      53.639803
Separated    51.533333
Single       51.778862
Widowed      61.519149
Name: Age, dtype: float64


In [60]:
df["Marital Status"] = df["Marital Status"].str.strip()
df["Marital_Lost"] = df["Marital Status"].isin(["Divorced", "Widowed", "Separated"]).astype(int)

In [61]:
df.columns = df.columns.str.strip()
df.columns

Index(['Age', 'Race', 'Marital Status', 'T Stage', 'N Stage', '6th Stage',
       'differentiate', 'Grade', 'A Stage', 'Tumor Size', 'Estrogen Status',
       'Progesterone Status', 'Regional Node Examined',
       'Reginol Node Positive', 'Survival Months', 'Status', 'Marital_Lost'],
      dtype='object')

In [62]:
y = df['Status']
y = (df["Status"] == "Dead").astype(int)
X = df.drop(columns = ['Survival Months' , 'Status' , 'Marital Status'])

In [63]:
y.value_counts(normalize=True)*100

Status
0    84.691849
1    15.308151
Name: proportion, dtype: float64

In [64]:
skewed  = ["Tumor Size", "Regional Node Examined", "Reginol Node Positive"]
numeric = ["Age"]
binary  = ["Marital_Lost"]
categorical = ["Race", "T Stage", "N Stage", "6th Stage", "differentiate",
               "Grade", "A Stage", "Estrogen Status", "Progesterone Status"]

In [65]:
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler , OneHotEncoder , FunctionTransformer

In [66]:
skewed_pipeline = Pipeline([
    ("Log" , FunctionTransformer(np.log1p)),
    ("Scale", StandardScaler())
])

numeric_pipeline = Pipeline([
    ('Scale' , StandardScaler())
])
categorical_pipeline = Pipeline([
    ("Ohe" , OneHotEncoder(drop="first" , handle_unknown="ignore" ))
])

In [67]:
trf = ColumnTransformer([
    ("skewed" , skewed_pipeline , skewed),
    ("numeric" , numeric_pipeline , numeric),
    ("categorical" , categorical_pipeline , categorical)
], remainder= "passthrough")

In [68]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("prep", trf),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

In [69]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
model.fit(X_train, y_train)

print(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))




0.7234222635481488


In [77]:
from sklearn.metrics import classification_report

probs = model.predict_proba(X_test)[:, 1]
y_pred = (probs >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["Alive", "Dead"]))

              precision    recall  f1-score   support

       Alive       0.91      0.71      0.79       682
        Dead       0.27      0.60      0.37       123

    accuracy                           0.69       805
   macro avg       0.59      0.65      0.58       805
weighted avg       0.81      0.69      0.73       805

